# QuickDash

## Objetivo

Diseñar un lenguaje de dominio específico, de sintaxis declarativa en español, que permita describir la visualización de datos tabulares (CSV) como tableros web, y traducir esa descripción a una página HTML con titulos, gráficos y  tablas.

## Objetivos específicos

- Definir una sintaxis y un conjunto de lexemas que permitan declarar fuentes de datos, indicadores, gráficos, tablas y filtros sin recurrir a la sintaxis de HTML ni JavaScript.
- Especificar formalmente la gramática del lenguaje y su análisis léxico y sintáctico mediante PLY.
- Validar la coherencia de los tableros definidos (referencias a fuentes, columnas y propiedades obligatorias). 
- Generar automáticamente el código web resultante (HTML + Chart.js) con apoyo de un agente de IA (Ollama).
- Reducir la barrera técnica para publicar visualizaciones, orientándolo a usuarios sin conocimientos de programación web.

## Alcance

**Incluye**:
- Declaración de una o más fuentes de datos en formato CSV.
- Gráficos de barras, líneas, torta y dispersión, con propiedades de ejes, color y límite de elementos.
- Tablas con selección de columnas y límite de filas.
- Comentarios y mensajes de error con número de línea por consola
- Traducción a un HTML que permita modificación de colores principales y/o datos que se presenten en el tablero.

**No incluye**: 
- Análisis o modelado de datos (relaciones entre tablas, medidas calculadas, transformaciones).
- Conexión a bases de datos, APIs o datos en tiempo real.
- Autenticación, publicación o alojamiento del tablero generado.
- Formatos de entrada distintos de CSV.
- A diferencia de herramientas de análisis como Power BI o Excel, este lenguaje no busca explorar ni modelar datos, sino describir de forma concisa cómo deben presentarse, y generar la visualización web correspondiente a partir de datos ya analizados y/o filtrados.

## Ejemplo

## Especificaciones léxicas

| Categoria | Tokens | Lexemas |
| :--- | :---: | ---: |
| Estructura | TABLERO, FUENTE, GRAFICO, TABLA, TITULO, SUBTITULO | tablero, grafico, tabla, fuente, titulo, subtitulo |
| Tipos de gráfico | BARRAS, LINEAS, TORTA, DISPERSION | barras, lineas, torta, dispersion |
| Propiedades | DATOS, EJE_X, EJE_Y, COLOR, LIMITE, COLUMNAS | eje_x, eje_y, columnas, limite, color |
| Otras reservadas | CSV | csv |
| Literales | STRING, NUMBER, HEXCOLOR, ID | ID: ventas, mes STRING: "Ventas Mensuales" NUMBER: 10 HEXCOLOR: #0000FF |
| Simbolos| LBRACE, RBRACE, LPAREN, RPAREN, DOSPUNTOS, IGUAL, COMA | {} () : = , |
| Ignorados| Espacios, saltos de líneas, comentarios // | //comentario |

## Especificaciones sintáticas

## Especificaciones semánticas

| Regla (GIC) | Acción semántica (AST) | Restricciones semánticas | Traducción a HTML |
| :--- | :--- | :--- | :--- |
| `tablero → TABLERO STRING { fuentes contenido }` | Crea el nodo raíz con título, lista de fuentes y lista de elementos. | Tiene al menos una fuente y un elemento, y las fuentes van antes del contenido (lo impone la gramática). | El `STRING` pasa a ser el `<title>` de la página. |
| `fuentes → fuentes fuente_decl \| fuente_decl` | Acumula las fuentes en una lista. | **(E)** No puede haber dos fuentes con el mismo nombre. | — |
| `contenido → contenido bloque \| bloque`<br>`bloque → titulo \| subtitulo \| grafico \| tabla` | Acumula los elementos en una lista ordenada. | Se conserva el orden de declaración. | La página sigue ese orden. Los gráficos consecutivos se agrupan en una misma fila. |
| `fuente_decl → FUENTE ID = CSV ( STRING )` | Registra nombre, ruta y línea. | **(E)** El archivo CSV existe, se puede leer y tiene encabezado y al menos una fila. Se detecta el separador y qué columnas son numéricas. **(A)** Fuente declarada pero no usada. | Sus datos alimentan los gráficos y tablas que la referencian. |
| `titulo_decl → TITULO STRING` | Nodo `titulo` con su texto. | Sin restricciones. | `<h1>` |
| `subtitulo_decl → SUBTITULO STRING` | Nodo `subtitulo` con su texto. | Sin restricciones. | `<h2>` |
| `grafico_decl → GRAFICO tipo_grafico STRING { propiedades }` | Nodo `grafico` con subtipo, título y lista de propiedades. | **(E)** Tiene las propiedades obligatorias, solo las permitidas para su tipo y ninguna repetida (ver tabla siguiente). | Tarjeta con `<h3>` (título), `<canvas>` y un gráfico de Chart.js. |
| `tipo_grafico → BARRAS \| LINEAS \| TORTA \| DISPERSION` | Guarda el subtipo. | Define qué propiedades se permiten y qué tipo de dato exige cada eje. | `bar`, `line`, `pie`, `scatter` de Chart.js. |
| `propiedades → propiedades propiedad \| propiedad`<br>`propiedad → DATOS : ID \| EJE_X : columna \| EJE_Y : columna \| COLOR : color \| LIMITE : NUMBER` | Acumula pares `(clave, valor)`. | `datos`: **(E)** fuente declarada. `eje_x`: **(E)** columna existente. `eje_y`: **(E)** columna existente y numérica. `color`: **(E)** nombre válido o `#RRGGBB`. `limite`: **(E)** entero positivo. | Cada fila del CSV es una barra, un punto o una porción. `limite` conserva las primeras $N$ filas. El nombre de `eje_y` es la leyenda. |
| `color → ID \| HEXCOLOR` | Guarda el valor del color. | **(E)** Uno de: `azul`, `verde`, `rojo`, `naranja`, `amarillo`, `violeta`, `gris`, o hexadecimal. | Se convierte a `rgba(...)`. |
| `tabla_decl → TABLA STRING { props_tabla }` | Nodo `tabla` con título y propiedades. | **(E)** Obligatorias: `datos` y `columnas`. Permitidas: `datos`, `columnas`, `limite`. Sin repetidas. | Tarjeta con `<h3>` y `<table>`. |
| `props_tabla → props_tabla prop_tabla \| prop_tabla`<br>`prop_tabla → DATOS : ID \| COLUMNAS : lista_columnas \| LIMITE : NUMBER` | Acumula pares `(clave, valor)`. | `datos`: **(E)** fuente declarada. `columnas`: **(E)** cada una existe en el CSV; **(A)** repetidas. `limite`: **(E)** entero positivo. | Los encabezados `<th>` salen de los nombres de columna. `limite` acota las filas. |
| `lista_columnas → lista_columnas , columna \| columna` | Lista ordenada de nombres de columna. | — | El orden de la lista es el orden de las columnas de la tabla. |
| `columna → ID \| STRING` | Guarda el nombre sin comillas (`mes` y `"mes"` son lo mismo). | **(E)** Debe existir en el CSV. Con comillas se admiten espacios, tildes y palabras reservadas (`"titulo"`). | — |


## Implementaciones

In [ ]:
%%file lexer.py
import sys
import ply.lex as lex


reservadas = {
    'tablero': 'TABLERO', 'fuente': 'FUENTE',
    'titulo': 'TITULO', 'subtitulo': 'SUBTITULO',
    'grafico': 'GRAFICO', 'tabla': 'TABLA',
    'barras': 'BARRAS', 'lineas': 'LINEAS',
    'torta': 'TORTA', 'dispersion': 'DISPERSION',
    'datos': 'DATOS', 'eje_x': 'EJE_X', 'eje_y': 'EJE_Y',
    'color': 'COLOR', 'limite': 'LIMITE', 'columnas': 'COLUMNAS',
    'csv': 'CSV',
}

tokens = [
    'ID', 'STRING', 'NUMBER', 'HEXCOLOR',
    'LBRACE', 'RBRACE', 'LPAREN', 'RPAREN',
    'DOSPUNTOS', 'IGUAL', 'COMA',
] + list(reservadas.values())

t_LBRACE = r'\{'
t_RBRACE = r'\}'
t_LPAREN = r'\('
t_RPAREN = r'\)'
t_DOSPUNTOS = r':'
t_IGUAL = r'='
t_COMA = r','

t_ignore = ' \t\r'

def t_COMMENT(t):
    r'//.*'
    pass

def t_HEXCOLOR(t):
    r'\#[0-9A-Fa-f]{6}'
    return t

def t_STRING(t):
    r'"[^"\n]*"'
    t.value = t.value[1:-1]
    return t

def t_NUMBER(t):
    r'\d+(\.\d+)?'
    t.value = float(t.value) if '.' in t.value else int(t.value)
    return t

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z_0-9]*'
    t.type = reservadas.get(t.value, 'ID')
    return t

def t_newline(t):
    r'\n+'
    t.lexer.lineno += len(t.value)

def t_error(t):
    print(f"[Error léxico] Línea {t.lexer.lineno}: carácter ilegal '{t.value[0]}'")
    t.lexer.skip(1)
    raise SyntaxError(f"Error léxico: carácter ilegal '{t.value[0]}'")

lexer = lex.lex()

In [ ]:
%%file parser_dash.py
import ply.yacc as yacc
from lexer import tokens, lexer   # 'tokens' es obligatorio importarlo

start = 'tablero'
errores = []
# Variables para pruebas semánticas (ej. tabla de símbolos simulada)
tabla_simbolos = set()
# ---------- estructura (fuentes primero) ----------
def p_tablero(p):
    'tablero : TABLERO STRING LBRACE fuentes contenido RBRACE'
    p[0] = {'tipo': 'tablero', 'titulo': p[2],
            'fuentes': p[4], 'elementos': p[5]}

def p_fuentes_varias(p):
    'fuentes : fuentes fuente_decl'
    p[0] = p[1] + [p[2]]

def p_fuentes_una(p):
    'fuentes : fuente_decl'
    p[0] = [p[1]]

def p_contenido_varios(p):
    'contenido : contenido bloque'
    p[0] = p[1] + [p[2]]

def p_contenido_uno(p):
    'contenido : bloque'
    p[0] = [p[1]]

def p_bloque(p):
    '''bloque : titulo_decl
              | subtitulo_decl
              | grafico_decl
              | tabla_decl'''
    p[0] = p[1]

# ---------- fuente, título, subtítulo ----------
def p_fuente_decl(p):
    'fuente_decl : FUENTE ID IGUAL CSV LPAREN STRING RPAREN'
    p[0] = {'tipo': 'fuente', 'nombre': p[2], 'ruta': p[6],
            'linea': p.lineno(1)}

def p_titulo_decl(p):
    'titulo_decl : TITULO STRING'
    p[0] = {'tipo': 'titulo', 'texto': p[2], 'linea': p.lineno(1)}

def p_subtitulo_decl(p):
    'subtitulo_decl : SUBTITULO STRING'
    p[0] = {'tipo': 'subtitulo', 'texto': p[2], 'linea': p.lineno(1)}

# ---------- gráfico ----------
def p_grafico_decl(p):
    'grafico_decl : GRAFICO tipo_grafico STRING LBRACE propiedades RBRACE'
    p[0] = {'tipo': 'grafico', 'subtipo': p[2], 'titulo': p[3],
            'propiedades': p[5], 'linea': p.lineno(1)}

def p_tipo_grafico(p):
    '''tipo_grafico : BARRAS
                    | LINEAS
                    | TORTA
                    | DISPERSION'''
    p[0] = p[1]

def p_propiedades_varias(p):
    'propiedades : propiedades propiedad'
    p[0] = p[1] + [p[2]]

def p_propiedades_una(p):
    'propiedades : propiedad'
    p[0] = [p[1]]

def p_propiedad_datos(p):
    'propiedad : DATOS DOSPUNTOS ID'
    p[0] = ('datos', p[3])

def p_propiedad_ejes(p):
    '''propiedad : EJE_X DOSPUNTOS columna
                 | EJE_Y DOSPUNTOS columna'''
    p[0] = (p[1], p[3])

def p_propiedad_color(p):
    'propiedad : COLOR DOSPUNTOS color'
    p[0] = ('color', p[3])

def p_propiedad_limite(p):
    'propiedad : LIMITE DOSPUNTOS NUMBER'
    p[0] = ('limite', p[3])

def p_color(p):
    '''color : ID
             | HEXCOLOR'''
    p[0] = p[1]

# ---------- tabla ----------
def p_tabla_decl(p):
    'tabla_decl : TABLA STRING LBRACE props_tabla RBRACE'
    p[0] = {'tipo': 'tabla', 'titulo': p[2],
            'propiedades': p[4], 'linea': p.lineno(1)}

def p_props_tabla_varias(p):
    'props_tabla : props_tabla prop_tabla'
    p[0] = p[1] + [p[2]]

def p_props_tabla_una(p):
    'props_tabla : prop_tabla'
    p[0] = [p[1]]

def p_prop_tabla_datos(p):
    'prop_tabla : DATOS DOSPUNTOS ID'
    p[0] = ('datos', p[3])

def p_prop_tabla_columnas(p):
    'prop_tabla : COLUMNAS DOSPUNTOS lista_columnas'
    p[0] = ('columnas', p[3])

def p_prop_tabla_limite(p):
    'prop_tabla : LIMITE DOSPUNTOS NUMBER'
    p[0] = ('limite', p[3])

def p_lista_columnas_varias(p):
    'lista_columnas : lista_columnas COMA columna'
    p[0] = p[1] + [p[3]]

def p_lista_columnas_una(p):
    'lista_columnas : columna'
    p[0] = [p[1]]

def p_columna(p):
    '''columna : ID
               | STRING'''
    p[0] = p[1]

# ---------- errores ----------
def p_error(p):
    if p:
        msg = (f"[Error sintáctico] Línea {p.lineno}: "
               f"token inesperado '{p.value}' ({p.type})")
    else:
        msg = ("[Error sintáctico] Fin de archivo inesperado "
               "(¿falta cerrar una llave?)")
    errores.append(msg)
    print(msg)
    raise SyntaxError(f"Error sintáctico: carácter ilegal '{p.value[0]}'")

parser = yacc.yacc(write_tables=False, debug=False)   #false para que no haga el debug y las tablas cada vez

In [ ]:
%%file semantica.py
import csv
import difflib
import math
import os

COLORES = {
    'azul': (54, 162, 235), 'verde': (75, 192, 192), 'rojo': (255, 99, 132),
    'naranja': (255, 159, 64), 'amarillo': (255, 205, 86),
    'violeta': (153, 102, 255), 'gris': (201, 203, 207),
}

_COMUNES = {'datos', 'eje_x', 'eje_y', 'color', 'limite'}
PERMITIDAS_GRAFICO = {
    'barras': _COMUNES, 'lineas': _COMUNES, 'dispersion': _COMUNES,
    'torta': _COMUNES - {'color'},
}
OBLIGATORIAS_GRAFICO = {'datos', 'eje_x', 'eje_y'}
OBLIGATORIAS_TABLA = {'datos', 'columnas'}
PERMITIDAS_TABLA = {'datos', 'columnas', 'limite'}


def a_numero(texto):
    """Convierte un texto a número (acepta coma o punto decimal) o devuelve None."""
    t = str(texto).strip().replace(',', '.')
    if not t or '_' in t:
        return None
    try:
        n = float(t)
    except ValueError:
        return None
    if math.isnan(n) or math.isinf(n):
        return None
    return int(n) if n.is_integer() else n


class Validador:
    def __init__(self, ast, base_dir='.'):
        self.ast = ast
        self.base_dir = base_dir
        self.fuentes = {}      # nombre -> {ruta, linea, columnas, numericas, filas}
        self.usadas = set()
        self.errores = []
        self.advertencias = []

    # ---------------- utilidades ----------------
    def _err(self, linea, msg):
        self.errores.append(f"[Error semántico] Línea {linea}: {msg}")

     # --- Para los test ---
    def _adv(self, linea, msg):
        self.advertencias.append(f"[Advertencia] Línea {linea}: {msg}")

    @staticmethod
    def _sugerir(valor, opciones):
        c = difflib.get_close_matches(str(valor), list(opciones), n=1)
        return f" ¿Quisiste decir '{c[0]}'?" if c else ""

    # ---------------- fuentes ----------------
    def _cargar_fuentes(self):
        for f in self.ast['fuentes']:
            nombre, linea = f['nombre'], f['linea']
            if nombre in self.fuentes:
                self._err(linea, f"la fuente '{nombre}' ya fue declarada")
                continue
            info = {'ruta': f['ruta'], 'linea': linea, 'columnas': None,
                    'numericas': set(), 'filas': []}
            self.fuentes[nombre] = info
            ruta = os.path.join(self.base_dir, f['ruta'])
            if not os.path.isfile(ruta):
                self._err(linea, f"no se encontró el archivo CSV '{f['ruta']}'")
                continue
            try:
                self._leer_csv(ruta, info)
            except (OSError, UnicodeDecodeError, csv.Error) as e:
                info['columnas'] = None
                self._err(linea, f"no se pudo leer '{f['ruta']}': {e}")
                continue
            if not info['filas']:
                self._err(linea, f"el CSV '{f['ruta']}' no tiene filas de datos")

    def _leer_csv(self, ruta, info):
        with open(ruta, newline='', encoding='utf-8-sig') as fh:
            muestra = fh.read(4096)
            fh.seek(0)
            try:
                dialecto = csv.Sniffer().sniff(muestra, delimiters=',;\t')
            except csv.Error:
                dialecto = csv.excel
            lector = csv.reader(fh, dialecto)
            cabecera = next(lector, None)
            if not cabecera:
                raise csv.Error('el archivo está vacío')
            cabecera = [c.strip() for c in cabecera]
            info['columnas'] = cabecera
            for fila in lector:
                fila = [c.strip() for c in fila]
                if any(fila):
                    info['filas'].append(dict(zip(cabecera, fila)))
        for col in cabecera:
            valores = [f.get(col, '') for f in info['filas'] if f.get(col, '')]
            if valores and all(a_numero(v) is not None for v in valores):
                info['numericas'].add(col)

    # ---------------- propiedades ----------------
    def _props(self, props, obligatorias, permitidas, ctx, linea):
        valores = {}
        for clave, valor in props:
            if clave not in permitidas:
                self._err(linea, f"{ctx}: la propiedad '{clave}' no está permitida")
            elif clave in valores:
                self._err(linea, f"{ctx}: la propiedad '{clave}' está repetida")
            else:
                valores[clave] = valor
        for falta in sorted(obligatorias - set(valores)):
            self._err(linea, f"{ctx}: falta la propiedad obligatoria '{falta}'")
        return valores

    def _comunes(self, v, ctx, linea):
        lim = v.get('limite')
        if lim is not None and (not isinstance(lim, int) or lim <= 0):
            self._err(linea, f"{ctx}: 'limite' debe ser un entero positivo")
        color = v.get('color')
        if color and not color.startswith('#') and color not in COLORES:
            self._err(linea, f"{ctx}: color '{color}' desconocido. "
                             f"Opciones: {', '.join(COLORES)} o #RRGGBB")

    def _fuente(self, v, ctx, linea):
        nombre = v.get('datos')
        if nombre is None:
            return None
        if nombre not in self.fuentes:
            self._err(linea, f"{ctx}: la fuente '{nombre}' no fue declarada."
                             + self._sugerir(nombre, self.fuentes))
            return None
        self.usadas.add(nombre)
        return nombre if self.fuentes[nombre]['columnas'] is not None else None

    # ---------------- elementos ----------------
    def _v_grafico(self, el):
        linea, tipo = el['linea'], el['subtipo']
        ctx = f"gráfico {tipo} '{el['titulo']}'"
        v = self._props(el['propiedades'], OBLIGATORIAS_GRAFICO,
                        PERMITIDAS_GRAFICO[tipo], ctx, linea)
        self._comunes(v, ctx, linea)
        nombre = self._fuente(v, ctx, linea)
        if nombre is None:
            return
        info = self.fuentes[nombre]
        for eje in ('eje_x', 'eje_y'):
            col = v.get(eje)
            if col is None:
                continue
            if col not in info['columnas']:
                self._err(linea, f"{ctx}: la columna '{col}' ({eje}) no existe "
                                 f"en la fuente '{nombre}'."
                                 + self._sugerir(col, info['columnas']))
            elif (eje == 'eje_y' or tipo == 'dispersion') \
                    and col not in info['numericas']:
                self._err(linea, f"{ctx}: la columna '{col}' ({eje}) debe ser numérica")
        x, y = v.get('eje_x'), v.get('eje_y')
        if tipo != 'dispersion' and x in info['columnas']:
            etiquetas = [f.get(x, '') for f in info['filas']]
            if len(set(etiquetas)) < len(etiquetas):
                self._adv(linea, f"{ctx}: 'eje_x' ({x}) tiene valores repetidos; "
                                 "cada fila se dibuja por separado")
        if tipo == 'torta' and y in info['numericas']:
            if any((a_numero(f.get(y, '')) or 0) < 0 for f in info['filas']):
                self._adv(linea, f"{ctx}: hay valores negativos en '{y}' "
                                 "y una torta no puede representarlos")

    def _v_tabla(self, el):
        linea = el['linea']
        ctx = f"tabla '{el['titulo']}'"
        v = self._props(el['propiedades'], OBLIGATORIAS_TABLA,
                        PERMITIDAS_TABLA, ctx, linea)
        self._comunes(v, ctx, linea)
        nombre = self._fuente(v, ctx, linea)
        if nombre is None:
            return
        info = self.fuentes[nombre]
        vistas = set()
        for col in v.get('columnas', []):
            if col in vistas:
                self._adv(linea, f"{ctx}: la columna '{col}' está repetida")
            vistas.add(col)
            if col not in info['columnas']:
                self._err(linea, f"{ctx}: la columna '{col}' no existe en la "
                                 f"fuente '{nombre}'."
                                 + self._sugerir(col, info['columnas']))

    # ---------------- entrada ----------------
    def ejecutar(self):
        self._cargar_fuentes()
        for el in self.ast['elementos']:
            if el['tipo'] == 'grafico':
                self._v_grafico(el)
            elif el['tipo'] == 'tabla':
                self._v_tabla(el)
        for nombre, info in self.fuentes.items():
            if nombre not in self.usadas:
                self._adv(info['linea'],
                          f"la fuente '{nombre}' se declara pero no se usa")


def validar(ast, base_dir='.'):
    v = Validador(ast, base_dir)
    v.ejecutar()
    # Si se acumularon errores -> excepción al final
    if v.errores:
        # Union de todos los mensajes de error acumulados con saltos de línea
        detalle_errores = "\n".join(v.errores)
        raise ValueError(f"\n{detalle_errores}")
    return v

In [ ]:
%%file generador.py
import json
import os
import re
from html import escape

from semantica import COLORES, a_numero

MODELO = os.environ.get('DASH_MODELO', 'qwen2.5:3b')
TEMPERATURAS = (0.2, 0.5, 0.8)      # un intento por valor, cada vez más variado permitiendo que el agente logre generar el ejemplo
MAX_INTENTOS = len(TEMPERATURAS)
PALETA = list(COLORES.values())

# ---------------------------------------------------------------
# Parte fija de la página (la escribe Python, no el modelo)
# Se opta por un estilo CSS fijo y una estructura HTML fija para la página. Evitando que el agente se trabe o decisa omitir su creación por concentrarse en los gráficos.  
PLANTILLA = """<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>__TITULO__</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
    <style>
        body { font-family: sans-serif; margin: 40px; background-color: #f4f7f6; color: #333; }
        h1 { margin-bottom: 30px; }
        .contenedor-graficos { display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 20px; margin-top: 5px}
        .tarjeta {
            background: white; padding: 20px; border-radius: 8px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05); flex: 1; min-width: 300px;
        }
        .tarjeta h3 { margin-top: 0; }
        .tarjeta canvas { max-width: 100%; }
        .tarjeta-tabla { margin-bottom: 20px; }
        .tabla-datos { width: 100%; border-collapse: collapse; margin-top: 15px; }
        .tabla-datos th, .tabla-datos td {
            padding: 12px 15px; text-align: left; border-bottom: 1px solid #e0e0e0;
        }
        .tabla-datos th { background-color: #f8f9fa; font-weight: 600; color: #555; }
        .tabla-datos tbody tr:hover { background-color: #fcfcfc; }
    </style>
</head>
<body>
__CUERPO__
</body>
</html>
"""

# ---------------------------------------------------------------
# Prompt: reglas cortas + UN ejemplo del tipo de elemento a generar
# ---------------------------------------------------------------
SISTEMA = """Sos un generador de HTML. Recibís UN elemento en JSON y devolvés \
únicamente el bloque HTML de ese elemento: sin explicaciones y sin bloques de markdown.

REGLAS
1. Seguí el ejemplo al pie de la letra: mismas etiquetas, mismas clases y mismos marcadores {{...}}.
2. Cambiá solo: el título del <h3> (campo "titulo").
3. Escribí un único bloque: una sola tarjeta, un solo <canvas> y un solo new Chart (o una sola tabla). No agregues otros gráficos ni otros elementos.
4. No escribas nombres de columnas, series, colores, números ni filas: todo eso son marcadores {{...}} que debés copiar tal cual.
5. No agregues <script src=...>: Chart.js ya está cargado."""

EJEMPLOS_GRAFICO = {
    'barras': (
        '{"n":1,"subtipo":"barras","titulo":"Título de ejemplo"}',
        '''<div class="tarjeta">
  <h3>Título de ejemplo</h3>
  <canvas id="grafico1"></canvas>
  <script>
    new Chart(document.getElementById('grafico1'), {
      type: 'bar',
      data: {
        labels: {{LABELS_1}},
        datasets: [{
          label: {{SERIE_1}},
          data: {{VALORES_1}},
          backgroundColor: {{FONDO_1}},
          borderColor: {{BORDE_1}},
          borderWidth: 1
        }]
      },
      options: { responsive: true, scales: { y: { beginAtZero: true } } }
    });
  </script>
</div>'''),
    'lineas': (
        '{"n":2,"subtipo":"lineas","titulo":"Título de ejemplo"}',
        '''<div class="tarjeta">
  <h3>Título de ejemplo</h3>
  <canvas id="grafico2"></canvas>
  <script>
    new Chart(document.getElementById('grafico2'), {
      type: 'line',
      data: {
        labels: {{LABELS_1}},
        datasets: [{
          label: {{SERIE_1}},
          data: {{VALORES_1}},
          backgroundColor: {{FONDO_1}},
          borderColor: {{BORDE_1}},
          borderWidth: 2,
          tension: 0.3
        }]
      },
      options: { responsive: true, scales: { y: { beginAtZero: true } } }
    });
  </script>
</div>'''),
    'torta': (
        '{"n":3,"subtipo":"torta","titulo":"Título de ejemplo"}',
        '''<div class="tarjeta">
  <h3>Título de ejemplo</h3>
  <canvas id="grafico3"></canvas>
  <script>
    new Chart(document.getElementById('grafico3'), {
      type: 'pie',
      data: {
        labels: {{LABELS_1}},
        datasets: [{
          label: {{SERIE_1}},
          data: {{VALORES_1}},
          backgroundColor: {{COLORES_1}}
        }]
      },
      options: { responsive: true }
    });
  </script>
</div>'''),
    'dispersion': (
        '{"n":4,"subtipo":"dispersion","titulo":"Título de ejemplo"}',
        '''<div class="tarjeta">
  <h3>Título de ejemplo</h3>
  <canvas id="grafico4"></canvas>
  <script>
    new Chart(document.getElementById('grafico4'), {
      type: 'scatter',
      data: {
        datasets: [{
          label: {{SERIE_1}},
          data: {{PUNTOS_1}},
          backgroundColor: {{FONDO_1}},
          borderColor: {{BORDE_1}}
        }]
      },
      options: { responsive: true }
    });
  </script>
</div>'''),
}

EJEMPLO_TABLA = (
    '{"tipo":"tabla","n":1,"titulo":"Título de ejemplo"}',
    '''<div class="tarjeta tarjeta-tabla">
  <h3>Título de ejemplo</h3>
  <table class="tabla-datos">
    <thead>
      <tr>{{ENCABEZADOS_1}}</tr>
    </thead>
    <tbody>{{FILAS_1}}</tbody>
  </table>
</div>''')

TIPO_JS = {'barras': 'bar', 'lineas': 'line', 'torta': 'pie', 'dispersion': 'scatter'}
MARCADOR = re.compile(r'\{\{([A-Z]+_\d+)\}\}')


class GeneracionFallida(Exception):
    def __init__(self, problemas, ultimo, elemento):
        super().__init__('; '.join(problemas))
        self.problemas = problemas
        self.ultimo = ultimo
        self.elemento = elemento



# ----------Plan: Json + prompt que recibe la IA (sin datos) + marcadores que rellena Python-------

def _json(obj):
    # el "</" se escapa para que un dato <script> en el csv no pueda cerrar el <script>
    return json.dumps(obj, ensure_ascii=False).replace('</', '<\\/')


def _rgba(color, alfa, indice):
    if color is None:
        rgb = PALETA[indice % len(PALETA)]
    elif color.startswith('#'):
        rgb = tuple(int(color[i:i + 2], 16) for i in (1, 3, 5))
    else:
        rgb = COLORES[color]
    return f"rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, {alfa})"


def _limitar(filas, limite):
    return filas[:limite] if limite else filas


def _item_grafico(el, val, n):
    props = dict(el['propiedades'])
    info = val.fuentes[props['datos']]
    filas = _limitar(info['filas'], props.get('limite'))
    x, y, sub = props['eje_x'], props['eje_y'], el['subtipo']
    item = {'n': n, 'subtipo': sub, 'titulo': el['titulo']}
    marc = {f'SERIE_{n}': _json(y)}
    if sub == 'dispersion':
        puntos = [{'x': a_numero(f.get(x, '')), 'y': a_numero(f.get(y, ''))}
                  for f in filas]
        marc[f'PUNTOS_{n}'] = _json([p for p in puntos
                                     if p['x'] is not None and p['y'] is not None])
    else:
        marc[f'LABELS_{n}'] = _json([f.get(x, '') for f in filas])
        marc[f'VALORES_{n}'] = _json([a_numero(f.get(y, '')) for f in filas])
    if sub == 'torta':
        marc[f'COLORES_{n}'] = _json([_rgba(None, 0.6, i) for i in range(len(filas))])
    else:
        color = props.get('color')
        alfa = 0.2 if sub == 'lineas' else 0.6
        marc[f'FONDO_{n}'] = _json(_rgba(color, alfa, n - 1))
        marc[f'BORDE_{n}'] = _json(_rgba(color, 1, n - 1))
    return item, marc


def construir_plan(ast, val):
    """Devuelve (plan, marcadores con los datos reales)."""
    plan, marcadores = [], {}
    n_graf = n_tabla = 0
    fila = None
    for el in ast['elementos']:
        if el['tipo'] == 'grafico':
            n_graf += 1
            item, marc = _item_grafico(el, val, n_graf)
            marcadores.update(marc)
            if fila is None:                 # gráficos consecutivos = una fila
                fila = {'tipo': 'fila_graficos', 'graficos': []}
                plan.append(fila)
            fila['graficos'].append(item)
            continue
        fila = None
        if el['tipo'] in ('titulo', 'subtitulo'):
            plan.append({'tipo': el['tipo'], 'texto': el['texto']})
        elif el['tipo'] == 'tabla':
            n_tabla += 1
            props = dict(el['propiedades'])
            info = val.fuentes[props['datos']]
            cols = props['columnas']
            filas = _limitar(info['filas'], props.get('limite'))
            marcadores[f'ENCABEZADOS_{n_tabla}'] = ''.join(
                f'<th>{escape(c)}</th>' for c in cols)
            marcadores[f'FILAS_{n_tabla}'] = ''.join(
                '<tr>' + ''.join(f"<td>{escape(f.get(c, ''))}</td>" for c in cols)
                + '</tr>' for f in filas)
            plan.append({'tipo': 'tabla', 'n': n_tabla, 'titulo': el['titulo']})
    return plan, marcadores



# ----------Prompt de un elemento-----------

def _ejemplo(item):
    """Ejemplo del tipo de elemento, ya numerado con el 'n' del elemento real."""
    n = item['n']
    if 'subtipo' in item:
        entrada, salida = EJEMPLOS_GRAFICO[item['subtipo']]
        tipo = f"GRÁFICO «{item['subtipo']}»"
    else:
        entrada, salida = EJEMPLO_TABLA
        tipo = 'TABLA'
    entrada = entrada.replace('"n":1', f'"n":{n}')
    salida = salida.replace('grafico1', f'grafico{n}')
    salida = re.sub(r'\{\{([A-Z]+)_1\}\}',
                    lambda m: '{{' + m.group(1) + f'_{n}' + '}}', salida)
    return f"EJEMPLO DE {tipo}\nEntrada: {entrada}\nSalida:\n{salida}"


def construir_prompt(item):
    return SISTEMA + '\n\n' + _ejemplo(item)



# ----------Llamada a Ollama-------------------

def _llamar(mensajes, tope, temperatura, etiqueta):
    import ollama   # se importa acá para poder probar el resto sin Ollama
    partes = []
    flujo = ollama.chat(
        model=MODELO, messages=mensajes, stream=True, keep_alive='10m',
        options={'temperature': temperatura, 'num_ctx': 4096,
                 'num_predict': tope, 'repeat_penalty': 1.0},
    )
    for trozo in flujo:
        partes.append(trozo['message']['content'])
        print(f"\r  [{etiqueta}] generando... ~{len(partes)} tokens", end='', flush=True)
    print()
    return ''.join(partes)


def _limpiar(texto):
    m = re.search(r'```(?:html)?\s*(.*?)```', texto, re.S)
    if m:
        texto = m.group(1)
    m = re.search(r'<body[^>]*>(.*?)</body>', texto, re.S | re.I)
    if m:
        texto = m.group(1)
    texto = re.sub(r'<script[^>]*\bsrc=[^>]*>\s*</script>', '', texto, flags=re.I)
    return texto.strip()


#------------- Verificación de un bloque-------------

def _contiene(texto_html, texto):
    return any(t in texto_html for t in (texto, escape(texto), escape(texto, quote=False)))


def _claves(item):
    """Marcadores que deben aparecer exactamente una vez en el bloque."""
    n = item['n']
    if 'subtipo' not in item:
        return [f'ENCABEZADOS_{n}', f'FILAS_{n}']
    sub = item['subtipo']
    if sub == 'dispersion':
        return [f'SERIE_{n}', f'PUNTOS_{n}', f'FONDO_{n}', f'BORDE_{n}']
    if sub == 'torta':
        return [f'SERIE_{n}', f'LABELS_{n}', f'VALORES_{n}', f'COLORES_{n}']
    return [f'SERIE_{n}', f'LABELS_{n}', f'VALORES_{n}', f'FONDO_{n}', f'BORDE_{n}']


def verificar_bloque(bloque, item):
    if not bloque:
        return ['la respuesta está vacía']
    problemas = []
    if not bloque.rstrip().endswith('</div>'):
        problemas.append('el bloque no termina en </div> (¿quedó cortado?)')
    if len(re.findall(r'<h3\b', bloque)) != 1:
        problemas.append('debe haber exactamente un <h3>')
    if not _contiene(bloque, item['titulo']):
        problemas.append(f"falta el título «{item['titulo']}»")

    if 'subtipo' in item:
        n, sub = item['n'], item['subtipo']
        if len(re.findall(r'<canvas\b', bloque)) != 1:
            problemas.append('debe haber exactamente un <canvas>')
        ids = re.findall(r'id=["\']grafico' + str(n) + r'["\']', bloque)
        if len(ids) != 1:
            problemas.append(f'debe haber un <canvas id="grafico{n}"> (hay {len(ids)})')
        if len(re.findall(r'new\s+Chart\s*\(', bloque)) != 1:
            problemas.append('debe haber exactamente una llamada a new Chart')
        for s, js in TIPO_JS.items():
            hay = len(re.findall(r'type:\s*["\']' + js + r'["\']', bloque))
            esperado = 1 if s == sub else 0
            if hay != esperado:
                problemas.append(f"hay {hay} gráfico(s) con type '{js}' y debería haber {esperado}")
    else:
        if len(re.findall(r'<table\b', bloque)) != 1:
            problemas.append('debe haber exactamente una <table>')

    esperados = _claves(item)
    hallados = MARCADOR.findall(bloque)
    for m in esperados:
        if hallados.count(m) != 1:
            problemas.append('el marcador {{' + m + '}} debe aparecer una vez '
                             f'(aparece {hallados.count(m)})')
    for m in sorted(set(hallados) - set(esperados)):
        problemas.append('marcador inexistente {{' + m + '}}')
    return problemas


# -------Generación y ensamblado--------

def _bloque(item):
    es_grafico = 'subtipo' in item
    etiqueta = f"gráfico {item['n']}" if es_grafico else f"tabla {item['n']}"
    tope = 600 if es_grafico else 300
    sistema = construir_prompt(item)
    pedido = 'ELEMENTO:\n' + json.dumps(item, ensure_ascii=False, separators=(',', ':'))
    nota, ultimo, problemas = '', '', []
    for intento, temp in enumerate(TEMPERATURAS, 1):
        print(f"{etiqueta}: intento {intento}/{MAX_INTENTOS} ({MODELO}, temperatura {temp})")
        mensajes = [{'role': 'system', 'content': sistema},
                    {'role': 'user', 'content': pedido + nota}]
        ultimo = _limpiar(_llamar(mensajes, tope, temp, etiqueta))
        problemas = verificar_bloque(ultimo, item)
        if not problemas:
            return ultimo
        print("  Problemas detectados:")
        for p in problemas:
            print(f"   - {p}")
        # el reintento NO recibe la respuesta anterior: solo la lista de problemas
        nota = ('\n\nEl intento anterior falló por: ' + '; '.join(problemas)
                + '. Devolvé un único bloque correcto.')
    raise GeneracionFallida(problemas, ultimo, etiqueta)


def _sustituir(cuerpo, marcadores):
    # una sola pasada: lo insertado no se vuelve a analizar
    return MARCADOR.sub(lambda m: marcadores.get(m.group(1), m.group(0)), cuerpo)


def generar_html(ast, plan, marcadores):
    cuerpo = []
    for it in plan:
        if it['tipo'] == 'titulo':
            cuerpo.append(f"<h1>{escape(it['texto'])}</h1>")
        elif it['tipo'] == 'subtitulo':
            cuerpo.append(f"<h2>{escape(it['texto'])}</h2>")
        elif it['tipo'] == 'fila_graficos':
            bloques = [_bloque(g) for g in it['graficos']]
            cuerpo.append('<div class="contenedor-graficos">\n'
                          + '\n'.join(bloques) + '\n</div>')
        elif it['tipo'] == 'tabla':
            cuerpo.append(_bloque(it))
    html = _sustituir('\n'.join(cuerpo), marcadores)
    return (PLANTILLA.replace('__TITULO__', escape(ast['titulo']))
                     .replace('__CUERPO__', html))

In [ ]:
%%file main.py
import json
import os
import sys

from lexer import lexer
import parser_dash
import semantica
import generador
from IPython.display import display, HTML, IFrame

def main():
    if len(sys.argv) < 2:
        print("Uso: python main.py archivo.dash")
        sys.exit(1)
        
    ruta = sys.argv[1]
    base_dir = os.path.dirname(os.path.abspath(ruta))   # los CSV se buscan acá
    base = os.path.splitext(ruta)[0]
    
    with open(ruta, encoding='utf-8') as f:
        codigo = f.read()
    
    #imprimo programa fuente
    print(f"=== Código fuente ({ruta}) ===")
    print(codigo)
    print("=" * 40)
    # Todo lo que tu código imprima con print() saldrá primero en consola.
    
    # Codigo guarda archivo HTML como f"{base}.html"
    archivo_html = f"{base}.html"
    
    """
    # --- Mostrar el resultado en Jupyter ---
    if os.path.exists(archivo_html):
        print(f"\n--- Vista previa de {archivo_html} ---")
        # Opción recomendada: IFrame aísla los estilos CSS/JS del tablero
        display(IFrame(src=archivo_html, width="100%", height=500))
    """
    # 1 y 2: análisis léxico y sintáctico
    lexer.lineno = 1
    ast = parser_dash.parser.parse(codigo, lexer=lexer)
    if ast is None or parser_dash.errores:
        print(f"\n{len(parser_dash.errores)} error(es) sintáctico(s). No se continúa.")
        sys.exit(1)

    # 3: análisis semántico
    val = semantica.validar(ast, base_dir)
    for adv in val.advertencias:
        print(adv)
    if val.errores:
        for err in val.errores:
            print(err)
        print(f"\n{len(val.errores)} error(es) semántico(s). No se genera código.")
        sys.exit(1)

    plan, marcadores = generador.construir_plan(ast, val)
    with open(base + '.ast.json', 'w', encoding='utf-8') as f:
        json.dump(ast, f, indent=2, ensure_ascii=False)
    with open(base + '.plan.json', 'w', encoding='utf-8') as f:
        json.dump(plan, f, indent=2, ensure_ascii=False)
    print(f"AST guardado en {base}.ast.json y plan para la IA en {base}.plan.json")

    # 4: generación con Ollama
    try:
        html = generador.generar_html(ast, plan, marcadores)
    except generador.GeneracionFallida as e:
        with open(base + '.fallido.txt', 'w', encoding='utf-8') as f:
            f.write(e.ultimo)
        print(f"\n[Error de generación] El modelo no pudo generar el {e.elemento} "
              f"tras {generador.MAX_INTENTOS} intentos. "
              f"Última respuesta en {base}.fallido.txt")
        sys.exit(1)
    except Exception as e:
        print(f"\n[Error de Ollama] {e}")
        print("Verificá que Ollama esté corriendo y que el modelo esté instalado (ollama list).")
        sys.exit(1)

    with open(base + '.html', 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"\nListo: {base}.html")


if __name__ == '__main__':
    main()

In [ ]:
!python main.py Ejemplos/minimo.dash

In [ ]:
!python main.py Ejemplos/medioCompleto.dash

In [ ]:
!python main.py Ejemplos/completo.dash

## Pruebas: TestSuite
Se realizan archivos de prueba buscando validar distintas partes del lenguaje creado para analizar su funcionamiento en su totalidad. Estos archivos de pruebas se encuentran en la carpeta "test_casos".

| Nro. / Archivo | Qué se evalúa (Objetivo) | Comportamiento Esperado |
| :--- | :--- | :--- |
| **Test_01. `valido_1.dash`** | Estructura básica correcta de un tablero con un gráfico, una tabla, titulo y subtitulo. | Pasa por el lexer, parser y semántica sin errores (`ok`). |
| **Test_02. `valido_2.dash`** | Consulta de columnas que sí existen en el archivo CSV de origen. | Pasa el pipeline completo correctamente (`ok`). |
| **Test_03. `valido_3.dash`** | Estructura con contenido tipo tabla unicamente. Una o más. | Pasa el parser y la validación semántica sin objeciones (`ok`). |
| **Test_04. `valido_4.dash`** | Declaración de más de una fuente. | Valida correctamente ambas fuentes y sus gráficos (`ok`). |
| **Test_05. `valido_5.dash`** | Archivo de prueba integral que combina todos los elementos válidos del lenguaje. | Supera el pipeline completo de punta a punta (`ok`). |
| **Test_06. `invalido_1_lexico.dash`** | Detección de **caracteres ilegales** no reconocidos por el lexer (ej. `@`). | El lexer frena el proceso y lanza `SyntaxError`, capturado por el test (`ok`). |
| **Test_07. `invalido_2_sintactico.dash`** | Errores de estructura, como **falta de cierre de llaves** (`}`). | El parser detecta fin de archivo inesperado, lanza `SyntaxError` (`ok`). |
| **Test_08. `invalido_3_sintactico.dash`** | Uso de **tokens inesperados** en medio de una definición de bloque porque dentro de contenido debe ir una fuente y un bloque. | El parser de PLY rechaza la gramática y lanza `SyntaxError` (`ok`). |
| **Test_09. `invalido_4_sintactico.dash`** | Errores en la disposición de tokens que violan la gramática esperada porque espera una fuente. | El parser detecta el fallo sintáctico y lanza `SyntaxError` (`ok`). |
| **Test_10. `invalido_5_semantico.dash`** | Inconsistencias de negocio, como **columnas inexistentes** en el CSV. | Supera a PLY pero el analizador semántico detecta el error, lanza `ValueError` (`ok`). |
| **Test_11. `sem_err.dash`** | Inconsistencias de negocio, como presencia de "color" en un grafico de tortas. | Supera a PLY pero el analizador semántico detecta el error, lanza `ValueError` (`ok`). |


In [ ]:
%%file test_traductor.py 
import os
import unittest

# Importa modulos ply y parser porque ellos contienen los codigos que atrapan los errores que utiliza la libreria unittest para hacer las pruebas. 
# Si existe un buen manejo de errores, entonces la prueba dará ok. 
from lexer import lexer
from parser_dash import parser
from semantica import validar

class TestArquitecturaCompilador(unittest.TestCase):

    def test_archivos_validos_pipeline_completo(self):
        """Verifica los 5 archivos válidos (Léxico + Sintaxis)."""
        carpeta = "test_casos/validos/"
        archivos = sorted([f for f in os.listdir(carpeta) if f.endswith(".dash")])

        self.assertEqual(
            len(archivos), 5, "Debe haber exactamente 5 archivos en 'validos/'."
        )

        for archivo in archivos:
            ruta = os.path.join(carpeta, archivo)
            with open(ruta, "r", encoding="utf-8") as f:
                codigo = f.read()

            with self.subTest(archivo=archivo):
                print(
                    f"\n[PROBANDO VÁLIDO] -> {archivo}"
                )  # Mensaje visual opcional en consola
                try:
                    resultado = parser.parse(codigo, lexer=lexer)
                    self.assertIsNotNone(
                        resultado, f"El archivo {archivo} devolvió None."
                    )
                except Exception as e:
                    self.fail(
                        f" El archivo válido '{archivo}' falló inesperadamente: {e}"
                    )
   
    def test_archivos_invalidos_deben_fallar(self):
        carpeta = "test_casos/invalidos/"
        archivos = sorted([f for f in os.listdir(carpeta) if f.endswith(".dash")])

        for archivo in archivos:
            ruta = os.path.join(carpeta, archivo)
            with open(ruta, "r", encoding="utf-8") as f:
                codigo = f.read()

            with self.subTest(archivo=archivo):
                try:
            # validar arbol, parser y semantica
                    arbol = parser.parse(codigo, lexer=lexer)
                    validar(arbol, base_dir="Ejemplos/")
                    # Si falta algun assertError, o ValueError 
                    self.fail(f" El archivo '{archivo}' pasó y DEBÍA FALLAR.")
                    
                except (SyntaxError, ValueError, Exception) as e:
                # Descripción exacta del error (léxico, sintáctico o semántico) en la terminal y el html de reportes
                    print(f"   ↳ {archivo} falló correctamente con: {e}")           
            


if __name__ == "__main__":
    unittest.main(verbosity=2)  # modo detallado automático

In [ ]:
!python test_traductor.py

## CONCLUSIONES

- Puntos importantes que fueron surgiendo durante la creación del lenguaje
    - Entender el alcance de mi lenguaje: a veces abarcar muchas funcionalidades puede llevar a entorpercer el codigo o incluso a, inconscientemente, copiar lenguajes o programas ya existentes.
    - Entender el alcance de Ollama y las diferencias entre los modelos que ofrece: es importante ir evaluando el tiempo y los resultados de cada respuesta dada por el agente. Este puede quedarse trabado debido a loops en el codigo, o utilizar mucho GPU que lo imposibilite a ser más eficiente. El modelo se debe escoger a conciencia del lenguaje creado, del resultado minimo esperado y de la capacidad de la máquina que se está utilizando. 
    - Entender cuál es el mejor plan para integrar la IA con el código propio: encontrar los prompts indicados para sacar el máximo provecho del agente y sus modelos es fundamental. Al inicio del trabajo práctico se empezó con unas instrucciones de rol/agente, como era insuficiente para que el agente comprenda bien su rol se agregaron reglas cortas. Sin embargo, aún habia problemas para que entienda cómo manejar el código de la manera correcta. Ahí entra la técnica few shot prompting, que consiste en incluir algunos ejemplos resueltos que el modelo pueda usar como plantilla para entender qué se está necesitando de él. 
- Finalmente el trabajo me permitió ver en profundidad todos los pasos involucrados en la creación de un lenguaje y especialmente me dio la oportunidad de crear algo que me resulta útil en mi día a día, con ayuda de la inclusión de la inteligencia artificial. 
    